In [1]:
import sys
import shutil
from importlib.metadata import version

print("Interprete:", sys.executable)
print("PyTorch:", version("torch"))
print("Pyannote Audio:", version("pyannote-audio"))
print("Hugging Face Hub:", version("huggingface-hub"))
print("FFmpeg:", shutil.which("ffmpeg"))

Interprete: c:\Users\acer\Desktop\ProgettoTesi\.venv\Scripts\python.exe
PyTorch: 2.13.0
Pyannote Audio: 4.0.7
Hugging Face Hub: 1.25.1
FFmpeg: C:\Users\acer\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffmpeg.EXE


In [2]:
import torch
import pyannote.audio
from pyannote.audio import Pipeline

print("Import completato correttamente")
print("PyTorch:", torch.__version__)
print("Pyannote Audio:", pyannote.audio.__version__)
print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("La diarizzazione verrà eseguita sulla CPU")

c:\Users\acer\Desktop\ProgettoTesi\.venv\lib\site-packages\pyannote\audio\core\io.py:48: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.13.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
            

Import completato correttamente
PyTorch: 2.13.0+cpu
Pyannote Audio: 4.0.7
CUDA disponibile: False
La diarizzazione verrà eseguita sulla CPU


c:\Users\acer\Desktop\ProgettoTesi\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Test preliminare della diarizzazione su ID89

In [1]:
from getpass import getpass
from huggingface_hub import login, whoami

hf_token = getpass(
    "Incolla il token Hugging Face che inizia con hf_: "
).strip()

if not hf_token:
    raise ValueError(
        "Non è stato inserito alcun token. "
        "Incolla il token nella casella e premi Invio."
    )

if not hf_token.startswith("hf_"):
    raise ValueError(
        "Il token non sembra valido: deve iniziare con hf_."
    )

login(
    token=hf_token,
    add_to_git_credential=False,
    skip_if_logged_in=False
)

utente = whoami(token=hf_token)

print("Accesso effettuato come:", utente["name"])

c:\Users\acer\Desktop\ProgettoTesi\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accesso effettuato come: LadyBug99


In [2]:
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=hf_token
)

if pipeline is None:
    raise RuntimeError(
        "Pipeline non caricata. Controlla il token e l'accesso al modello."
    )

print("Pipeline caricata correttamente")

c:\Users\acer\Desktop\ProgettoTesi\.venv\lib\site-packages\pyannote\audio\core\io.py:48: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.13.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
            

Pipeline caricata correttamente


In [3]:
from pathlib import Path

import soundfile as sf
import torch


def carica_segmento_audio(
    percorso,
    canale=0,
    inizio_secondi=0,
    durata_secondi=60
):
    audio, sample_rate = sf.read(
        Path(percorso),
        always_2d=True,
        dtype="float32"
    )

    if canale >= audio.shape[1]:
        raise ValueError(
            f"Canale {canale} non disponibile: "
            f"il file ha {audio.shape[1]} canale/i."
        )

    inizio = int(inizio_secondi * sample_rate)
    fine = min(
        inizio + int(durata_secondi * sample_rate),
        len(audio)
    )

    segnale = audio[inizio:fine, canale]

    return {
        "waveform": torch.from_numpy(segnale).unsqueeze(0),
        "sample_rate": sample_rate
    }


audio_id89_sinistro = carica_segmento_audio(
    "file_wav/ID89.StudioRuggi.wav",
    canale=0,
    inizio_secondi=0,
    durata_secondi=60
)

print("Forma:", audio_id89_sinistro["waveform"].shape)
print("Sample rate:", audio_id89_sinistro["sample_rate"])

Forma: torch.Size([1, 2880000])
Sample rate: 48000


In [4]:
output_id89_sinistro = pipeline(audio_id89_sinistro)

print("Diarizzazione completata")

c:\Users\acer\Desktop\ProgettoTesi\.venv\lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1861.)
  std = sequences.std(dim=-1, correction=1)


Diarizzazione completata


In [5]:
diarizzazione = output_id89_sinistro.speaker_diarization

for segmento, _, speaker in diarizzazione.itertracks(
    yield_label=True
):
    print(
        f"{speaker}: "
        f"{segmento.start:.2f} s - {segmento.end:.2f} s"
    )

SPEAKER_01: 1.25 s - 2.21 s
SPEAKER_00: 3.14 s - 3.64 s
SPEAKER_01: 5.04 s - 5.73 s
SPEAKER_01: 6.02 s - 7.00 s
SPEAKER_00: 7.66 s - 8.11 s
SPEAKER_01: 9.45 s - 10.93 s
SPEAKER_00: 11.13 s - 11.40 s
SPEAKER_01: 13.04 s - 16.33 s
SPEAKER_01: 17.94 s - 19.10 s
SPEAKER_01: 20.89 s - 23.35 s
SPEAKER_01: 25.24 s - 26.15 s
SPEAKER_01: 26.22 s - 27.06 s
SPEAKER_00: 27.23 s - 27.55 s
SPEAKER_01: 28.11 s - 31.54 s
SPEAKER_01: 32.48 s - 34.93 s
SPEAKER_01: 35.89 s - 36.73 s
SPEAKER_01: 36.84 s - 39.06 s
SPEAKER_01: 40.09 s - 41.80 s
SPEAKER_01: 41.88 s - 42.93 s
SPEAKER_01: 43.96 s - 45.71 s
SPEAKER_00: 49.26 s - 52.43 s
SPEAKER_01: 53.51 s - 54.66 s
SPEAKER_00: 55.08 s - 55.53 s
SPEAKER_01: 57.02 s - 59.97 s


In [6]:
from IPython.display import Audio, display


def ascolta_segmento(audio_dict, inizio, fine, descrizione=""):
    waveform = audio_dict["waveform"][0].detach().cpu().numpy()
    sample_rate = audio_dict["sample_rate"]

    campione_inizio = int(inizio * sample_rate)
    campione_fine = int(fine * sample_rate)

    segmento = waveform[campione_inizio:campione_fine]

    print(f"{descrizione}: {inizio:.2f} s - {fine:.2f} s")
    display(Audio(segmento, rate=sample_rate))

In [7]:
ascolta_segmento(
    audio_id89_sinistro,
    49.26,
    52.43,
    "SPEAKER_00"
)

ascolta_segmento(
    audio_id89_sinistro,
    7.66,
    8.11,
    "SPEAKER_00"
)

SPEAKER_00: 49.26 s - 52.43 s


SPEAKER_00: 7.66 s - 8.11 s


In [8]:
ascolta_segmento(
    audio_id89_sinistro,
    13.04,
    16.33,
    "SPEAKER_01"
)

ascolta_segmento(
    audio_id89_sinistro,
    28.11,
    31.54,
    "SPEAKER_01"
)

ascolta_segmento(
    audio_id89_sinistro,
    57.02,
    59.97,
    "SPEAKER_01"
)

SPEAKER_01: 13.04 s - 16.33 s


SPEAKER_01: 28.11 s - 31.54 s


SPEAKER_01: 57.02 s - 59.97 s


In [9]:
from pathlib import Path
import pandas as pd

mappa_ruoli = {
    "SPEAKER_00": "paziente",
    "SPEAKER_01": "voce_registrata"
}

righe = []

for segmento, _, speaker in (
    output_id89_sinistro
    .speaker_diarization
    .itertracks(yield_label=True)
):
    righe.append({
        "recording_id": "ID89",
        "canale": "sinistro",
        "inizio_secondi": round(segmento.start, 3),
        "fine_secondi": round(segmento.end, 3),
        "durata_secondi": round(segmento.end - segmento.start, 3),
        "speaker": speaker,
        "ruolo": mappa_ruoli.get(speaker, "non_identificato")
    })

df_segmenti_id89 = pd.DataFrame(righe)

cartella_output = Path("risultati/diarizzazione")
cartella_output.mkdir(parents=True, exist_ok=True)

percorso_output = (
    cartella_output /
    "ID89_sinistro_0_60_segmenti.csv"
)

df_segmenti_id89.to_csv(
    percorso_output,
    index=False
)

display(df_segmenti_id89)

print("\nDurata complessiva per ruolo:")
print(
    df_segmenti_id89
    .groupby("ruolo")["durata_secondi"]
    .sum()
)

print("\nFile salvato in:", percorso_output)

,recording_id,canale,inizio_secondi,fine_secondi,durata_secondi,speaker,ruolo
0,ID89,sinistro,1.246,2.208,0.962,SPEAKER_01,voce_registrata
1,ID89,sinistro,3.136,3.642,0.506,SPEAKER_00,paziente
2,ID89,sinistro,5.043,5.735,0.692,SPEAKER_01,voce_registrata
3,ID89,sinistro,6.022,7.000,0.979,SPEAKER_01,voce_registrata
4,ID89,sinistro,7.658,8.114,0.456,SPEAKER_00,paziente
5,ID89,sinistro,9.447,10.932,1.485,SPEAKER_01,voce_registrata
6,ID89,sinistro,11.135,11.405,0.270,SPEAKER_00,paziente
7,ID89,sinistro,13.042,16.332,3.291,SPEAKER_01,voce_registrata
8,ID89,sinistro,17.935,19.100,1.164,SPEAKER_01,voce_registrata
9,ID89,sinistro,20.888,23.352,2.464,SPEAKER_01,voce_registrata



Durata complessiva per ruolo:
ruolo
paziente            5.181
voce_registrata    30.341
Name: durata_secondi, dtype: float64

File salvato in: risultati\diarizzazione\ID89_sinistro_0_60_segmenti.csv


In [10]:
audio_id89_destro = carica_segmento_audio(
    "file_wav/ID89.StudioRuggi.wav",
    canale=1,
    inizio_secondi=0,
    durata_secondi=60
)

print("Forma:", audio_id89_destro["waveform"].shape)
print("Sample rate:", audio_id89_destro["sample_rate"])

output_id89_destro = pipeline(audio_id89_destro)

print("Diarizzazione del canale destro completata")

Forma: torch.Size([1, 2880000])
Sample rate: 48000
Diarizzazione del canale destro completata


In [11]:
righe_destro = []

for segmento, _, speaker in (
    output_id89_destro
    .speaker_diarization
    .itertracks(yield_label=True)
):
    righe_destro.append({
        "inizio_secondi": round(segmento.start, 3),
        "fine_secondi": round(segmento.end, 3),
        "durata_secondi": round(segmento.end - segmento.start, 3),
        "speaker": speaker
    })

df_id89_destro = pd.DataFrame(righe_destro)

display(df_id89_destro)

print("\nDurata complessiva per speaker:")
print(
    df_id89_destro
    .groupby("speaker")["durata_secondi"]
    .sum()
)

,inizio_secondi,fine_secondi,durata_secondi,speaker
0,1.246,2.039,0.793,SPEAKER_00
1,2.039,2.225,0.186,SPEAKER_01
2,2.225,2.258,0.034,SPEAKER_00
3,3.119,3.608,0.489,SPEAKER_00
4,5.060,5.718,0.658,SPEAKER_01
5,6.005,7.000,0.996,SPEAKER_01
6,7.692,8.080,0.388,SPEAKER_00
7,9.447,10.898,1.451,SPEAKER_01
8,11.135,11.422,0.287,SPEAKER_00
9,13.177,16.349,3.172,SPEAKER_01



Durata complessiva per speaker:
speaker
SPEAKER_00     7.071
SPEAKER_01    28.435
Name: durata_secondi, dtype: float64


In [12]:
segmenti_lunghi = (
    df_id89_destro
    .sort_values("durata_secondi", ascending=False)
    .groupby("speaker")
    .head(3)
)

display(segmenti_lunghi)

,inizio_secondi,fine_secondi,durata_secondi,speaker
9,13.177,16.349,3.172,SPEAKER_01
24,49.272,52.428,3.156,SPEAKER_00
27,57.001,59.971,2.970,SPEAKER_01
11,20.888,23.369,2.481,SPEAKER_01
10,17.952,19.066,1.114,SPEAKER_00
0,1.246,2.039,0.793,SPEAKER_00


In [14]:
# SPEAKER_00: segmento più lungo
ascolta_segmento(
    audio_id89_destro,
    49.272,
    52.428,
    "SPEAKER_00 - canale destro"
)

# SPEAKER_00: secondo segmento
ascolta_segmento(
    audio_id89_destro,
    17.952,
    19.066,
    "SPEAKER_00 - canale destro"
)

# SPEAKER_01: segmento più lungo
ascolta_segmento(
    audio_id89_destro,
    13.177,
    16.349,
    "SPEAKER_01 - canale destro"
)

# SPEAKER_01: altro segmento lungo
ascolta_segmento(
    audio_id89_destro,
    57.001,
    59.971,
    "SPEAKER_01 - canale destro"
)

SPEAKER_00 - canale destro: 49.27 s - 52.43 s


SPEAKER_00 - canale destro: 17.95 s - 19.07 s


SPEAKER_01 - canale destro: 13.18 s - 16.35 s


SPEAKER_01 - canale destro: 57.00 s - 59.97 s


In [15]:
import pandas as pd
from pathlib import Path

audit_id89 = pd.DataFrame([
    {
        "recording_id": "ID89",
        "canale": "sinistro",
        "inizio_secondi": 49.260,
        "fine_secondi": 52.430,
        "speaker": "SPEAKER_00",
        "ruolo_ascoltato": "paziente"
    },
    {
        "recording_id": "ID89",
        "canale": "sinistro",
        "inizio_secondi": 13.040,
        "fine_secondi": 16.330,
        "speaker": "SPEAKER_01",
        "ruolo_ascoltato": "voce_registrata"
    },
    {
        "recording_id": "ID89",
        "canale": "destro",
        "inizio_secondi": 49.272,
        "fine_secondi": 52.428,
        "speaker": "SPEAKER_00",
        "ruolo_ascoltato": "paziente"
    },
    {
        "recording_id": "ID89",
        "canale": "destro",
        "inizio_secondi": 17.952,
        "fine_secondi": 19.066,
        "speaker": "SPEAKER_00",
        "ruolo_ascoltato": "voce_registrata"
    },
    {
        "recording_id": "ID89",
        "canale": "destro",
        "inizio_secondi": 13.177,
        "fine_secondi": 16.349,
        "speaker": "SPEAKER_01",
        "ruolo_ascoltato": "voce_registrata"
    }
])

audit_id89["esito"] = audit_id89.apply(
    lambda r: (
        "confusione"
        if r["canale"] == "destro" and r["speaker"] == "SPEAKER_00"
        else "coerente"
    ),
    axis=1
)

cartella_output = Path("risultati/diarizzazione")
cartella_output.mkdir(parents=True, exist_ok=True)

percorso_audit = cartella_output / "ID89_audit_manuale_canali.csv"

audit_id89.to_csv(
    percorso_audit,
    index=False
)

display(audit_id89)

print("File salvato in:", percorso_audit)

,recording_id,canale,inizio_secondi,fine_secondi,speaker,ruolo_ascoltato,esito
0,ID89,sinistro,49.260,52.430,SPEAKER_00,paziente,coerente
1,ID89,sinistro,13.040,16.330,SPEAKER_01,voce_registrata,coerente
2,ID89,destro,49.272,52.428,SPEAKER_00,paziente,confusione
3,ID89,destro,17.952,19.066,SPEAKER_00,voce_registrata,confusione
4,ID89,destro,13.177,16.349,SPEAKER_01,voce_registrata,coerente


File salvato in: risultati\diarizzazione\ID89_audit_manuale_canali.csv


## Diarizzazione automatica dell’intero dataset

Dopo la verifica preliminare su ID89, la pipeline viene applicata automaticamente a tutte le registrazioni, analizzando separatamente i canali.

In [16]:
from pathlib import Path
from scipy.signal import resample_poly
from math import gcd

import gc
import re
import numpy as np
import pandas as pd
import soundfile as sf
import torch


AUDIO_DIR = Path("file_wav")
OUTPUT_DIR = Path("risultati/diarizzazione_batch")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def estrai_patient_id(nome_file):
    risultato = re.search(
        r"ID(\d+)",
        nome_file,
        flags=re.IGNORECASE
    )

    if risultato:
        return risultato.group(1)

    return None


def nome_canale(indice_canale, numero_canali):
    if numero_canali == 1:
        return "mono"

    if indice_canale == 0:
        return "sinistro"

    if indice_canale == 1:
        return "destro"

    return f"canale_{indice_canale + 1}"


def carica_canale_16k(
    percorso,
    indice_canale,
    target_sr=16000
):
    audio, sample_rate = sf.read(
        percorso,
        always_2d=True,
        dtype="float32"
    )

    if indice_canale >= audio.shape[1]:
        raise ValueError(
            f"Canale {indice_canale} non disponibile."
        )

    segnale = audio[:, indice_canale]

    if not np.isfinite(segnale).all():
        raise ValueError(
            "Il segnale contiene valori non validi."
        )

    if sample_rate != target_sr:
        divisore = gcd(sample_rate, target_sr)

        segnale = resample_poly(
            segnale,
            target_sr // divisore,
            sample_rate // divisore
        ).astype("float32")

    waveform = torch.from_numpy(
        segnale.copy()
    ).unsqueeze(0)

    return {
        "waveform": waveform,
        "sample_rate": target_sr
    }


print("Funzioni per il batch definite correttamente.")

Funzioni per il batch definite correttamente.


In [17]:
file_wav = sorted(AUDIO_DIR.glob("*.wav"))

if not file_wav:
    raise FileNotFoundError(
        f"Nessun file WAV trovato nella cartella: {AUDIO_DIR.resolve()}"
    )

print("File WAV trovati:", len(file_wav))
print("\nPrime registrazioni individuate:")

for percorso in file_wav[:5]:
    informazioni = sf.info(percorso)

    print(
        f"- {percorso.name} | "
        f"canali: {informazioni.channels} | "
        f"durata: {informazioni.duration:.2f} secondi"
    )

# Per il primo test utilizziamo soltanto due registrazioni
file_da_analizzare = file_wav[:2]

print("\nFile selezionati per la prova:")
for percorso in file_da_analizzare:
    print("-", percorso.name)

File WAV trovati: 93

Prime registrazioni individuate:
- ID1.StudioRuggi-006.wav | canali: 2 | durata: 329.19 secondi
- ID10.StudioRuggi.wav | canali: 2 | durata: 203.16 secondi
- ID101.StudioRuggi.wav | canali: 2 | durata: 283.65 secondi
- ID103.StudioRuggi.wav | canali: 1 | durata: 524.16 secondi
- ID104.StudioRuggi.wav | canali: 2 | durata: 479.13 secondi

File selezionati per la prova:
- ID1.StudioRuggi-006.wav
- ID10.StudioRuggi.wav


In [18]:
def carica_segmento_canale_16k(
    percorso,
    indice_canale,
    inizio_secondi=0,
    durata_secondi=60,
    target_sr=16000
):
    informazioni = sf.info(percorso)
    sample_rate_originale = informazioni.samplerate

    frame_inizio = int(inizio_secondi * sample_rate_originale)
    numero_frame = int(durata_secondi * sample_rate_originale)

    audio, sample_rate = sf.read(
        percorso,
        start=frame_inizio,
        frames=numero_frame,
        always_2d=True,
        dtype="float32"
    )

    if audio.size == 0:
        raise ValueError("Il segmento audio selezionato è vuoto.")

    if indice_canale >= audio.shape[1]:
        raise ValueError(
            f"Canale {indice_canale} non disponibile: "
            f"il file contiene {audio.shape[1]} canale/i."
        )

    segnale = audio[:, indice_canale]

    if not np.isfinite(segnale).all():
        raise ValueError("Il segnale contiene valori non validi.")

    if sample_rate != target_sr:
        divisore = gcd(sample_rate, target_sr)

        segnale = resample_poly(
            segnale,
            target_sr // divisore,
            sample_rate // divisore
        ).astype("float32")

    return {
        "waveform": torch.from_numpy(segnale.copy()).unsqueeze(0),
        "sample_rate": target_sr
    }


print("Funzione per i segmenti di prova definita correttamente.")

Funzione per i segmenti di prova definita correttamente.


In [19]:
TEST_OUTPUT_DIR = OUTPUT_DIR / "test_2_file_60_secondi"
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

riepilogo_test = []

for numero_file, percorso_audio in enumerate(
    file_da_analizzare,
    start=1
):
    informazioni = sf.info(percorso_audio)
    numero_canali = informazioni.channels

    print(
        f"\n[{numero_file}/{len(file_da_analizzare)}] "
        f"{percorso_audio.name}"
    )

    for indice_canale in range(numero_canali):
        etichetta_canale = nome_canale(
            indice_canale,
            numero_canali
        )

        print(
            f"  {etichetta_canale}: "
            "diarizzazione dei primi 60 secondi..."
        )

        audio_input = None
        risultato = None

        try:
            audio_input = carica_segmento_canale_16k(
                percorso=percorso_audio,
                indice_canale=indice_canale,
                inizio_secondi=0,
                durata_secondi=60
            )

            with torch.inference_mode():
                risultato = pipeline(audio_input)

            righe = []

            for segmento, _, speaker in (
                risultato
                .speaker_diarization
                .itertracks(yield_label=True)
            ):
                righe.append({
                    "patient_id": estrai_patient_id(
                        percorso_audio.name
                    ),
                    "recording_id": percorso_audio.stem,
                    "nome_file": percorso_audio.name,
                    "canale": etichetta_canale,
                    "inizio_secondi": round(
                        segmento.start, 3
                    ),
                    "fine_secondi": round(
                        segmento.end, 3
                    ),
                    "durata_secondi": round(
                        segmento.end - segmento.start,
                        3
                    ),
                    "speaker": speaker,
                    "ruolo": "da_definire"
                })

            df_risultato = pd.DataFrame(righe)

            nome_csv = (
                f"{percorso_audio.stem}"
                f"__{etichetta_canale}"
                f"__0_60.csv"
            )

            percorso_csv = TEST_OUTPUT_DIR / nome_csv

            df_risultato.to_csv(
                percorso_csv,
                index=False
            )

            numero_speaker = (
                df_risultato["speaker"].nunique()
                if not df_risultato.empty
                else 0
            )

            print(
                f"  {etichetta_canale}: completato — "
                f"{len(df_risultato)} segmenti, "
                f"{numero_speaker} speaker"
            )

            riepilogo_test.append({
                "nome_file": percorso_audio.name,
                "canale": etichetta_canale,
                "stato": "completato",
                "numero_segmenti": len(df_risultato),
                "numero_speaker": numero_speaker,
                "errore": None
            })

        except Exception as errore:
            print(
                f"  {etichetta_canale}: ERRORE — {errore}"
            )

            riepilogo_test.append({
                "nome_file": percorso_audio.name,
                "canale": etichetta_canale,
                "stato": "errore",
                "numero_segmenti": None,
                "numero_speaker": None,
                "errore": str(errore)
            })

        finally:
            audio_input = None
            risultato = None
            gc.collect()


df_riepilogo_test = pd.DataFrame(riepilogo_test)

df_riepilogo_test.to_csv(
    TEST_OUTPUT_DIR / "riepilogo_test.csv",
    index=False
)

print("\nTest terminato.")
display(df_riepilogo_test)

print(
    "\nRisultati salvati in:",
    TEST_OUTPUT_DIR
)


[1/2] ID1.StudioRuggi-006.wav
  sinistro: diarizzazione dei primi 60 secondi...
  sinistro: completato — 20 segmenti, 2 speaker
  destro: diarizzazione dei primi 60 secondi...
  destro: completato — 20 segmenti, 2 speaker

[2/2] ID10.StudioRuggi.wav
  sinistro: diarizzazione dei primi 60 secondi...
  sinistro: completato — 20 segmenti, 2 speaker
  destro: diarizzazione dei primi 60 secondi...
  destro: completato — 20 segmenti, 2 speaker

Test terminato.


,nome_file,canale,stato,numero_segmenti,numero_speaker,errore
0,ID1.StudioRuggi-006.wav,sinistro,completato,20,2,None
1,ID1.StudioRuggi-006.wav,destro,completato,20,2,None
2,ID10.StudioRuggi.wav,sinistro,completato,20,2,None
3,ID10.StudioRuggi.wav,destro,completato,20,2,None



Risultati salvati in: risultati\diarizzazione_batch\test_2_file_60_secondi


In [20]:
from time import perf_counter

BATCH_DIR = OUTPUT_DIR / "completo"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

PERCORSO_STATO = BATCH_DIR / "stato_elaborazione.csv"

COLONNE_SEGMENTI = [
    "patient_id",
    "recording_id",
    "nome_file",
    "canale",
    "inizio_secondi",
    "fine_secondi",
    "durata_secondi",
    "speaker",
    "ruolo"
]


# Recupera l'eventuale registro di una precedente esecuzione
if PERCORSO_STATO.exists():
    df_stato = pd.read_csv(PERCORSO_STATO)
else:
    df_stato = pd.DataFrame(columns=[
        "nome_file",
        "canale",
        "stato",
        "numero_segmenti",
        "numero_speaker",
        "tempo_secondi",
        "errore"
    ])


def aggiorna_stato(nuova_riga):
    """
    Aggiorna il registro sostituendo l'eventuale riga precedente
    relativa allo stesso file e allo stesso canale.
    """
    global df_stato

    if not df_stato.empty:
        da_mantenere = ~(
            (df_stato["nome_file"] == nuova_riga["nome_file"]) &
            (df_stato["canale"] == nuova_riga["canale"])
        )

        df_stato = df_stato.loc[da_mantenere].copy()

    df_stato = pd.concat(
        [
            df_stato,
            pd.DataFrame([nuova_riga])
        ],
        ignore_index=True
    )

    percorso_temporaneo = (
        PERCORSO_STATO.parent /
        "stato_elaborazione.tmp.csv"
    )

    df_stato.to_csv(
        percorso_temporaneo,
        index=False
    )

    percorso_temporaneo.replace(PERCORSO_STATO)


print("Registrazioni da elaborare:", len(file_wav))
print("Cartella dei risultati:", BATCH_DIR)

Registrazioni da elaborare: 93
Cartella dei risultati: risultati\diarizzazione_batch\completo


In [22]:
for numero_file, percorso_audio in enumerate(file_wav, start=1):

    try:
        informazioni = sf.info(percorso_audio)
        numero_canali = informazioni.channels

    except Exception as errore:
        print(
            f"\n[{numero_file}/{len(file_wav)}] "
            f"{percorso_audio.name}: errore di lettura — {errore}"
        )
        continue

    print(
        f"\n[{numero_file}/{len(file_wav)}] "
        f"{percorso_audio.name} | "
        f"{numero_canali} canale/i | "
        f"{informazioni.duration:.2f} secondi"
    )

    for indice_canale in range(numero_canali):

        etichetta_canale = nome_canale(
            indice_canale,
            numero_canali
        )

        nome_csv = (
            f"{percorso_audio.stem}"
            f"__{etichetta_canale}.csv"
        )

        percorso_csv = BATCH_DIR / nome_csv
        percorso_temporaneo = (
            BATCH_DIR /
            f"{percorso_audio.stem}"
            f"__{etichetta_canale}.tmp.csv"
        )

        # I risultati già completati non vengono ricalcolati
        if percorso_csv.exists():
            print(
                f"  {etichetta_canale}: "
                "già completato, salto."
            )
            continue

        if percorso_temporaneo.exists():
            percorso_temporaneo.unlink()

        audio_input = None
        risultato = None
        inizio_elaborazione = perf_counter()

        try:
            print(
                f"  {etichetta_canale}: "
                "diarizzazione completa in corso..."
            )

            audio_input = carica_canale_16k(
                percorso_audio,
                indice_canale
            )

            with torch.inference_mode():
                risultato = pipeline(audio_input)

            righe = []

            for segmento, _, speaker in (
                risultato
                .speaker_diarization
                .itertracks(yield_label=True)
            ):
                righe.append({
                    "patient_id": estrai_patient_id(
                        percorso_audio.name
                    ),
                    "recording_id": percorso_audio.stem,
                    "nome_file": percorso_audio.name,
                    "canale": etichetta_canale,
                    "inizio_secondi": round(
                        segmento.start, 3
                    ),
                    "fine_secondi": round(
                        segmento.end, 3
                    ),
                    "durata_secondi": round(
                        segmento.end - segmento.start,
                        3
                    ),
                    "speaker": speaker,
                    "ruolo": "da_definire"
                })

            df_segmenti = pd.DataFrame(
                righe,
                columns=COLONNE_SEGMENTI
            )

            df_segmenti.to_csv(
                percorso_temporaneo,
                index=False
            )

            # Il file definitivo viene creato soltanto
            # quando l'elaborazione è terminata correttamente
            percorso_temporaneo.replace(percorso_csv)

            numero_speaker = (
                df_segmenti["speaker"].nunique()
                if not df_segmenti.empty
                else 0
            )

            tempo_impiegato = (
                perf_counter() - inizio_elaborazione
            )

            aggiorna_stato({
                "nome_file": percorso_audio.name,
                "canale": etichetta_canale,
                "stato": "completato",
                "numero_segmenti": len(df_segmenti),
                "numero_speaker": numero_speaker,
                "tempo_secondi": round(tempo_impiegato, 2),
                "errore": None
            })

            print(
                f"  {etichetta_canale}: completato — "
                f"{len(df_segmenti)} segmenti, "
                f"{numero_speaker} speaker, "
                f"{tempo_impiegato / 60:.1f} minuti"
            )

        except KeyboardInterrupt:
            tempo_impiegato = (
                perf_counter() - inizio_elaborazione
            )

            aggiorna_stato({
                "nome_file": percorso_audio.name,
                "canale": etichetta_canale,
                "stato": "interrotto",
                "numero_segmenti": None,
                "numero_speaker": None,
                "tempo_secondi": round(tempo_impiegato, 2),
                "errore": "Elaborazione interrotta manualmente"
            })

            if percorso_temporaneo.exists():
                percorso_temporaneo.unlink()

            print("\nElaborazione interrotta.")
            raise

        except Exception as errore:
            tempo_impiegato = (
                perf_counter() - inizio_elaborazione
            )

            aggiorna_stato({
                "nome_file": percorso_audio.name,
                "canale": etichetta_canale,
                "stato": "errore",
                "numero_segmenti": None,
                "numero_speaker": None,
                "tempo_secondi": round(tempo_impiegato, 2),
                "errore": str(errore)
            })

            if percorso_temporaneo.exists():
                percorso_temporaneo.unlink()

            print(
                f"  {etichetta_canale}: ERRORE — {errore}"
            )

        finally:
            audio_input = None
            risultato = None

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


print("\nDiarizzazione batch terminata.")

print("\nRiepilogo degli stati:")
print(df_stato["stato"].value_counts(dropna=False))

print("\nRegistro salvato in:", PERCORSO_STATO)


[1/93] ID1.StudioRuggi-006.wav | 2 canale/i | 329.19 secondi
  sinistro: già completato, salto.
  destro: già completato, salto.

[2/93] ID10.StudioRuggi.wav | 2 canale/i | 203.16 secondi
  sinistro: già completato, salto.
  destro: già completato, salto.

[3/93] ID101.StudioRuggi.wav | 2 canale/i | 283.65 secondi
  sinistro: già completato, salto.
  destro: già completato, salto.

[4/93] ID103.StudioRuggi.wav | 1 canale/i | 524.16 secondi
  mono: già completato, salto.

[5/93] ID104.StudioRuggi.wav | 2 canale/i | 479.13 secondi
  sinistro: già completato, salto.
  destro: già completato, salto.

[6/93] ID108.StudioRuggi.wav | 2 canale/i | 12.03 secondi
  sinistro: già completato, salto.
  destro: già completato, salto.

[7/93] ID11.StudioRuggi.wav | 2 canale/i | 160.22 secondi
  sinistro: già completato, salto.
  destro: già completato, salto.

[8/93] ID112.StudioRuggi.wav | 2 canale/i | 372.35 secondi
  sinistro: già completato, salto.
  destro: già completato, salto.

[9/93] ID114.